# 9 WorkFlow Analista Jr, futuro=JULIO

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Jul 17 20:35:34 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671157,35.9,1479352,79.1,1479352,79.1
Vcells,1242552,9.5,8388608,64.0,1978639,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
PARAM$semillas <- c(100019, 500029, 777977, 778777, 999961)

PARAM$semilla_primigenia <- PARAM$semillas[1]

PARAM$experimento <- 9113
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [ ]:
# se intenta corregir el data drifting utilizando algunos indices financieros

PCA

In [27]:
excluir <- c("numero_de_cliente", "foto_mes", "clase_ternaria")

PCA_Reduccion <- function(pca_varianza_objetivo, pca_meses_fit) {

  cat("inicio PCA_Reduccion()\n")

  campos_pca <- setdiff(colnames(dataset), excluir)

  for (campo in campos_pca) {
    dataset[is.na(get(campo)), (campo) := 0]
  }

  datos_fit <- as.matrix(dataset[foto_mes %in% pca_meses_fit, campos_pca, with = FALSE])
  modelo_pca <- prcomp(datos_fit, center = TRUE, scale. = TRUE)

  varianza_explicada <- cumsum(modelo_pca$sdev^2) / sum(modelo_pca$sdev^2)
  n_componentes <- min(which(varianza_explicada >= pca_varianza_objetivo))

  cat("PCA: se retienen", n_componentes, "de", length(campos_pca), "variables\n")

  datos_todos <- as.matrix(dataset[, campos_pca, with = FALSE])
  componentes <- predict(modelo_pca, newdata = datos_todos)[, 1:n_componentes]

  dataset[, (campos_pca) := NULL]
  for (i in 1:n_componentes) {
    dataset[, paste0("PC", i) := componentes[, i]]
  }

  cat("fin PCA_Reduccion()\n")
}

In [28]:
PARAM$PCA$varianza_objetivo <- 0.95
PARAM$PCA$meses_fit <- c(202101, 202102, 202103)

PCA_Reduccion(
  pca_varianza_objetivo = PARAM$PCA$varianza_objetivo,
  pca_meses_fit = PARAM$PCA$meses_fit
)

# verificacion rapida antes de seguir con el resto del notebook
ncol(dataset)
colnames(dataset)

inicio PCA_Reduccion()
PCA: se retienen 37 de 52 variables
fin PCA_Reduccion()


[1] 40

[1] "numero_de_cliente" "foto_mes"          "clase_ternaria"   
 [4] "PC1"               "PC2"               "PC3"              
 [7] "PC4"               "PC5"               "PC6"              
[10] "PC7"               "PC8"               "PC9"              
[13] "PC10"              "PC11"              "PC12"             
[16] "PC13"              "PC14"              "PC15"             
[19] "PC16"              "PC17"              "PC18"             
[22] "PC19"              "PC20"              "PC21"             
[25] "PC22"              "PC23"              "PC24"             
[28] "PC25"              "PC26"              "PC27"             
[31] "PC28"              "PC29"              "PC30"             
[34] "PC31"              "PC32"              "PC33"             
[37] "PC34"              "PC35"              "PC36"             
[40] "PC37"

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 201901, 202103 ]  donde se consideran el 100% de los CONTINUA

In [29]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [30]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [31]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [44]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [45]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x, dtrain, dvalidate) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  param_completo$seed <- PARAM$semilla_primigenia

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [34]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(256, 384, 512),
  min_data_in_leaf= c(64, 256, 512),
  feature_fraction= c(0.5, 0.8)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 60 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [40]:
if(!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

In [47]:
lista_mejores <- list()
tb_grid_base <- copy(tb_nueva)

for (i in seq_along(PARAM$semillas)) {
  
  PARAM$semilla_primigenia <- PARAM$semillas[i]
  cat("Grid Search para semilla:", PARAM$semilla_primigenia, "(", i, "de", length(PARAM$semillas), ")\n")
  
  
  # dtrain
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]
  
  # undersampling de los CONTINUA
  dataset[, fold_train :=  foto_mes %in% PARAM$trainingstrategy$training &
            (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
               azar < PARAM$trainingstrategy$training_pct) ]
  
  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_train == TRUE, clase01],
    free_raw_data = TRUE
  )

  cat("Semilla primigenia: ", PARAM$semilla_primigenia)
  cat("Hash:", digest::digest(dataset[fold_train == TRUE, azar]), "\n")
  
  dvalidate <- lgb.Dataset(
    data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
    label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
    free_raw_data= TRUE,
    reference = dtrain
  )
    
  tb_iteracion <- copy(tb_grid_base)
  tb_iteracion[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(.SD, dtrain, dvalidate), by = 1:nrow(tb_iteracion)]
  
  # 1 txt por semilla  
  nombre_archivo <- sprintf("tb_grid_search_%02d.txt", i)
  fwrite( 
    tb_iteracion,
    file = nombre_archivo,
    sep = "\t",
    append = FALSE
  )
  
  # resultados
  setorder(tb_iteracion, -AUC)
  mejor_configuracion <- tb_iteracion[1]
  mejor_configuracion[, semilla_utilizada := PARAM$semilla_primigenia]
  
  lista_mejores[[i]] <- mejor_configuracion
  
  #limpieza de memoria
  rm(dtrain, dvalidate)
  gc() 
}

# consolidación de resultados en tabla
tb_mejores_por_semilla <- rbindlist(lista_mejores)
PARAM$out$lgbm$mejores_hiperparametros <- tb_mejores_por_semilla

print(PARAM$out$lgbm$mejores_hiperparametros)

Grid Search para semilla: 100019 ( 1 de 5 )
Semilla primigenia:  100019Hash: cbe3e3757632f3d6c8713f4c46e4266d 


Fri Jul 17 21:57:19 2026  256, 64, 0.5 niter 468 AUC 0.892124448008529

Fri Jul 17 21:57:39 2026  256, 64, 0.8 niter 259 AUC 0.895583223299897

Fri Jul 17 21:58:05 2026  256, 256, 0.5 niter 409 AUC 0.895363694112424

Fri Jul 17 21:58:31 2026  256, 256, 0.8 niter 240 AUC 0.893503764180487

Fri Jul 17 21:58:56 2026  256, 512, 0.5 niter 391 AUC 0.896331681103514

Fri Jul 17 21:59:15 2026  256, 512, 0.8 niter 209 AUC 0.896247187714354

Fri Jul 17 21:59:41 2026  384, 64, 0.5 niter 332 AUC 0.892286598503545

Fri Jul 17 22:00:09 2026  384, 64, 0.8 niter 342 AUC 0.893115939524241

Fri Jul 17 22:01:06 2026  384, 256, 0.5 niter 968 AUC 0.894101977375743

Fri Jul 17 22:01:32 2026  384, 256, 0.8 niter 292 AUC 0.895345412815496

Fri Jul 17 22:01:51 2026  384, 512, 0.5 niter 153 AUC 0.892598071860859

Fri Jul 17 22:02:13 2026  384, 512, 0.8 niter 180 AUC 0.895540208483597

Fri Jul 17 22:02:48 2026  512, 64, 0.5 niter 429 AUC 0.893073231956629

Fri Jul 17 22:03:31 2026  512, 64, 0.8 niter 511 AUC 0.8

Grid Search para semilla: 500029 ( 2 de 5 )
Semilla primigenia:  500029Hash: 7e672695f48aee6beb34b33553d837ba 


Fri Jul 17 22:06:54 2026  256, 64, 0.5 niter 414 AUC 0.892988354506609

Fri Jul 17 22:07:17 2026  256, 64, 0.8 niter 371 AUC 0.892305187049161

Fri Jul 17 22:07:50 2026  256, 256, 0.5 niter 613 AUC 0.894413220296541

Fri Jul 17 22:08:12 2026  256, 256, 0.8 niter 352 AUC 0.89481679144804

Fri Jul 17 22:08:39 2026  256, 512, 0.5 niter 325 AUC 0.894673767183843

Fri Jul 17 22:09:09 2026  256, 512, 0.8 niter 498 AUC 0.89737924550476

Fri Jul 17 22:09:37 2026  384, 64, 0.5 niter 420 AUC 0.893485559695732

Fri Jul 17 22:09:59 2026  384, 64, 0.8 niter 246 AUC 0.891845312575612

Fri Jul 17 22:10:39 2026  384, 256, 0.5 niter 623 AUC 0.894774314316944

Fri Jul 17 22:11:01 2026  384, 256, 0.8 niter 220 AUC 0.895087477542041

Fri Jul 17 22:11:24 2026  384, 512, 0.5 niter 231 AUC 0.895303780618292

Fri Jul 17 22:11:55 2026  384, 512, 0.8 niter 345 AUC 0.897458515666227

Fri Jul 17 22:12:44 2026  512, 64, 0.5 niter 689 AUC 0.894251146613697

Fri Jul 17 22:13:21 2026  512, 64, 0.8 niter 394 AUC 0.895

Grid Search para semilla: 777977 ( 3 de 5 )
Semilla primigenia:  777977Hash: 0e1d22bf8cfb6884525669f08f098ee0 


...

Grid Search para semilla: 778777 ( 4 de 5 )
Semilla primigenia:  778777Hash: 6e4c35a8ade795f55387491676a70db8 


Fri Jul 17 22:26:46 2026  256, 64, 0.5 niter 583 AUC 0.895196627638402

Fri Jul 17 22:27:09 2026  256, 64, 0.8 niter 393 AUC 0.89460832321333

Fri Jul 17 22:28:01 2026  256, 256, 0.5 niter 1212 AUC 0.893715688962936

Fri Jul 17 22:28:20 2026  256, 256, 0.8 niter 286 AUC 0.896011067097737

Fri Jul 17 22:28:53 2026  256, 512, 0.5 niter 477 AUC 0.895971816077863

Fri Jul 17 22:29:15 2026  256, 512, 0.8 niter 311 AUC 0.894427507360526

Fri Jul 17 22:29:44 2026  384, 64, 0.5 niter 424 AUC 0.893820614389839

Fri Jul 17 22:30:11 2026  384, 64, 0.8 niter 331 AUC 0.89206814468648

Fri Jul 17 22:30:50 2026  384, 256, 0.5 niter 577 AUC 0.894343398032226

Fri Jul 17 22:31:17 2026  384, 256, 0.8 niter 281 AUC 0.89583286285878

Fri Jul 17 22:31:38 2026  384, 512, 0.5 niter 212 AUC 0.89514961858916

Fri Jul 17 22:32:02 2026  384, 512, 0.8 niter 219 AUC 0.89553874905233

Fri Jul 17 22:32:41 2026  512, 64, 0.5 niter 513 AUC 0.897068847517853

Fri Jul 17 22:33:11 2026  512, 64, 0.8 niter 325 AUC 0.89478

Grid Search para semilla: 999961 ( 5 de 5 )
Semilla primigenia:  999961Hash: b3a88038b755e46604313787206e4a1e 


Fri Jul 17 22:35:40 2026  256, 64, 0.5 niter 427 AUC 0.893394844520642

Fri Jul 17 22:36:02 2026  256, 64, 0.8 niter 357 AUC 0.892209402270721

Fri Jul 17 22:36:24 2026  256, 256, 0.5 niter 347 AUC 0.893224935996258

Fri Jul 17 22:36:56 2026  256, 256, 0.8 niter 479 AUC 0.894359067715306

Fri Jul 17 22:37:12 2026  256, 512, 0.5 niter 192 AUC 0.892142652493285

Fri Jul 17 22:37:35 2026  256, 512, 0.8 niter 303 AUC 0.896535924668767

Fri Jul 17 22:38:01 2026  384, 64, 0.5 niter 351 AUC 0.891957996031883

Fri Jul 17 22:38:34 2026  384, 64, 0.8 niter 495 AUC 0.891275366259639

Fri Jul 17 22:39:14 2026  384, 256, 0.5 niter 654 AUC 0.896470480698253

Fri Jul 17 22:39:43 2026  384, 256, 0.8 niter 353 AUC 0.89303689979929

Fri Jul 17 22:40:05 2026  384, 512, 0.5 niter 216 AUC 0.894407075322784

Fri Jul 17 22:40:55 2026  384, 512, 0.8 niter 725 AUC 0.895503108204538

Fri Jul 17 22:41:23 2026  512, 64, 0.5 niter 342 AUC 0.892936736727049

Fri Jul 17 22:41:50 2026  512, 64, 0.8 niter 273 AUC 0.89

   num_leaves min_data_in_leaf feature_fraction       AUC num_iterations
        <num>            <num>            <num>     <num>          <int>
1:        256              512              0.5 0.8963317            391
2:        512              512              0.5 0.8980457            627
3:        256              512              0.5 0.8965107            344
4:        512               64              0.5 0.8970688            513
5:        256              512              0.8 0.8965359            303
   semilla_utilizada
               <num>
1:            100019
2:            500029
3:            777977
4:            778777
5:            999961


In [64]:
resultados_finales <- list()

PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)
dataset[, fold_final_train := foto_mes %in% as.character(PARAM$trainingstrategy$final_train)]

PARAM$trainingstrategy$future <- c(202107)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

for (i in seq_len(nrow(PARAM$out$lgbm$mejores_hiperparametros))) {
  fila <- PARAM$out$lgbm$mejores_hiperparametros[i]
  PARAM$semilla_primigenia <- fila$semilla_utilizada
  cat("Modelo final semilla:", PARAM$semilla_primigenia, "(", i, "de", nrow(PARAM$out$lgbm$mejores_hiperparametros), ")\n")

  # dfinal_train
  dfinal_train <- lgb.Dataset(
    data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
    label= dataset[fold_final_train == TRUE, clase01],
    free_raw_data= FALSE
  )

  # armo param_final para esta semilla
  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  fijos$seed <- PARAM$semilla_primigenia

  param_final <- c(
    fijos,
    list(
      num_leaves = fila$num_leaves,
      min_data_in_leaf = fila$min_data_in_leaf,
      feature_fraction = fila$feature_fraction,
      num_iterations = fila$num_iterations
    )
  )

  # entreno
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
  final_model <- lgb.train(
    data= dfinal_train,
    param= param_final,
    verbose= -100
  )

  lgb.save(final_model, sprintf("modelo_%02d.txt", i))

  # importancia
  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite(tb_importancia, file= sprintf("impo_%02d.txt", i), sep= "\t")

  # prediccion
  dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]

  prediccion <- predict(
    final_model,
    data.matrix(dfuture[, campos_buenos, with= FALSE])
  )

  tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)]
  tb_prediccion[, prob := prediccion]

  fwrite(tb_prediccion, file= sprintf("prediccion_%02d.txt", i), sep= "\t")

  tb_prediccion[, clase_ternaria := dfuture$clase_ternaria]
  tb_prediccion[, ganancia := -0.025]
  tb_prediccion[clase_ternaria=="BAJA+2", ganancia := 0.975]

  setorder(tb_prediccion, -prob)
  tb_prediccion[, gan_acum := cumsum(ganancia)]
  tb_prediccion[, gan_suavizada := frollmean(
    x= gan_acum, n= 400, align= "center", na.rm= TRUE, has.nf= TRUE
  )]

  resultado <- list()
  resultado$semilla <- PARAM$semilla_primigenia
  resultado$ganancia_suavizada_max <- max(tb_prediccion$gan_suavizada, na.rm= TRUE)
  resultado$envios <- which.max(tb_prediccion$gan_suavizada)

  fwrite(tb_prediccion, file= sprintf("ganancias_%02d.txt", i), sep= "\t")

  tb_prediccion[, envios := .I]

  pdf(sprintf("curva_de_ganancia_%02d.pdf", i))
  plot(
    x= tb_prediccion$envios,
    y= tb_prediccion$gan_acum,
    type= "l",
    col= "gray",
    xlim= c(0, 10000),
    ylim= c(0, 15000000),
    main= paste0("Semilla ", PARAM$semilla_primigenia, " | gan prom= ",
                 as.integer(resultado$ganancia_suavizada_max),
                 " envios= ", resultado$envios),
    xlab= "Envios",
    ylab= "Ganancia",
    panel.first= grid()
  )
  dev.off()

  resultados_finales[[i]] <- as.data.table(resultado)

  rm(dfinal_train, final_model)
  gc()
}

tb_resultados_finales <- rbindlist(resultados_finales)
PARAM$resultado <- tb_resultados_finales

require("yaml")
write_yaml(PARAM, file= "PARAM.yml")

print(tb_resultados_finales)

Modelo final semilla: 100019 ( 1 de 5 )
Modelo final semilla: 500029 ( 2 de 5 )
Modelo final semilla: 777977 ( 3 de 5 )
Modelo final semilla: 778777 ( 4 de 5 )
Modelo final semilla: 999961 ( 5 de 5 )
   semilla ganancia_suavizada_max envios
     <num>                  <num>  <int>
1:  100019                77.6775   3214
2:  500029                79.9750   3116
3:  777977                83.9800   2852
4:  778777                77.1100   3363
5:  999961                81.2900   2570


la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Curva de Ganancia

Genero las salidas

In [ ]:
# asigno clase ternaria
tb_prediccion[, clase_ternaria := dfuture$clase_ternaria ]

# asigno ganancias
tb_prediccion[, ganancia := -0.025 ]
tb_prediccion[clase_ternaria=="BAJA+2", ganancia := 0.975 ]

# reordeno, y acumulada
setorder( tb_prediccion, -prob )
tb_prediccion[, gan_acum := cumsum(ganancia)]

# media movil de ancho 400
tb_prediccion[,
  gan_suavizada := frollmean(
    x= gan_acum,
    n= 400,
    align= "center",
    na.rm= TRUE,
    hasNA= TRUE
  )
]


In [ ]:
tb_prediccion[, sum(!is.na(gan_suavizada) )]
tb_prediccion[, .N]

In [ ]:
# Este es la ganancia a reportar
resultado <- list()
resultado$ganancia_suavizada_max <- max( tb_prediccion$gan_suavizada, na.rm=TRUE )
options(digits= 8)
resultado$envios <- which.max( tb_prediccion$gan_suavizada)
resultado

In [ ]:
# grabo las ganancias
fwrite( tb_prediccion,
  file= "ganancias.txt",
  sep= "\t"
)

In [ ]:
# genero el grafico

tb_prediccion[, envios:= .I]

pdf("curva_de_ganancia.pdf")

plot(
  x= tb_prediccion$envios,
  y= tb_prediccion$gan_acum,
  type= "l",
  col= "gray",
  xlim= c(0, 10000),
  ylim= c(0, 15000000),
  main= paste0("Mejor gan prom= ", as.integer(resultado$ganancia_suavizada_max), " envios= ", resultado$envios),
  xlab= "Envios",
  ylab= "Ganancia",
  panel.first= grid()
)

dev.off()

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

PARAM$resultado <- resultado

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")